In [3]:
from abc import ABC, abstractmethod
from functools import wraps

In [6]:
# This is our Abstraction (the contract)
class DataReader(ABC):
    def __init_subclass__(cls):
        if "connect" in cls.__dict__:
            original_connect = cls.__dict__["connect"]

            @wraps(original_connect)
            def wrapped_connect(self, *args, **kwargs):
                print("Running required set up before connection...")
                return original_connect(self, *args, **kwargs)
            setattr(cls, "connect", wrapped_connect)

        super().__init_subclass__()

    @abstractmethod
    def connect(self):
        """Establish connection to the data source."""
        pass

    @abstractmethod
    def fetch_data(self, query: str):
        """Fetch and return the data."""
        pass


# Implementation 1: Local Database
class PostgresReader(DataReader):
    def connect(self):
        print("Connecting to Postgres via psycopg2...")

    def fetch_data(self, query: str):
        print(f"Executing '{query}' on Postgres...")
        return {"data": "postgres_results"}


# Implementation 2: Cloud Storage
class S3ParquetReader(DataReader):
    def connect(self):
        print("Authenticating with AWS S3...")

    def fetch_data(self, query: str):
        print(f"Filtering Parquet files in S3 for '{query}'...")
        return {"data": "s3_results"}


def run_analytics(reader: DataReader):
    # The business logic relies on the abstraction, not the concrete class.
    reader.connect()
    data = reader.fetch_data("SELECT * FROM trades")
    print("Processing:", data)

In [10]:
# You can swap the implementation without changing the analytics logic
db_reader = S3ParquetReader()
run_analytics(db_reader)

Running required set up before connection...
Authenticating with AWS S3...
Filtering Parquet files in S3 for 'SELECT * FROM trades'...
Processing: {'data': 's3_results'}
